# <font color="steelblue">COVID-19 (México)</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

## <font color="steelblue">Objetivos del proyecto</font>

A partir de datos clínicos, demográficos y de comorbilidades de pacientes con COVID-19 en México (Gobierno Federal, 2020), construir, comparar y **desplegar** un clasificador que estime el **riesgo del paciente**. Al terminar, debéis ser capaces de:

* **Definir vosotros la variable objetivo** (no viene dada) y justificar la elección.
* Detectar y evitar la **fuga de datos** distinguiendo las variables **disponibles en el momento de la predicción** de las que son un **resultado** del proceso.
* Limpiar y **recodificar** correctamente (faltantes 97/98/99, binarios 1/2, fechas).
* **Comparar** varias familias de clasificadores con metodología sólida (CV, métricas adecuadas, sin fugas).
* Tratar el **desequilibrio** con **ponderación de muestras** y **remuestreo**, y medir su efecto.
* **Optimizar hiperparámetros**, **combinar modelos** si aporta, **interpretar** y **desplegar**.


## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

Los datos proceden del registro epidemiológico abierto de la **Secretaría de Salud de México** (datos.gob.mx), difundido en Kaggle por *meirnizri*. Se trata de un **registro administrativo poblacional**, no de una cohorte de investigación: cada fila corresponde a una persona atendida por el Sistema Nacional de Salud durante la pandemia, con su información demográfica, sus comorbilidades previas y el desenlace de su atención. El conjunto completo contiene 1.048.575 registros de pacientes,  descritos por **21 variables**. La versión original publicada por el gobierno mexicano incluía 40 variables y los nombres estaban en español;  la que utilizamos es una selección ya traducida.

Este origen administrativo explica dos rasgos que condicionan todo el análisis: las variables son casi todas **categóricas codificadas con números**, y la ausencia de dato no se representa con un valor vacío, sino con **códigos numéricos** que hay que interpretar.

### <font color="steelblue">Diccionario de variables</font>

**Bloque 1 — Demografía y clasificación diagnóstica**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `sex` | Nominal | 1 = mujer, 2 = hombre | Sexo del paciente. |
| `age` | Numérica continua | años | Edad. Es la **única variable genuinamente continua** del conjunto. |
| `classification` | Ordinal parcial | 1–3, ≥4 | Resultado del diagnóstico. Los valores **1, 2 y 3** indican **COVID-19 confirmado** con distinto grado de certeza o severidad; los valores **iguales o superiores a 4** indican que el paciente **no es portador** o que la prueba fue **inconcluyente**. |
| `patient_type` | Nominal | 1 = alta domiciliaria, 2 = hospitalización | Tipo de atención recibida: si el paciente volvió a casa (atención ambulatoria) o si requirió ingreso. |

**Bloque 2 — Gestión hospitalaria y resultado**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `usmr` | Nominal | 1, 2, 3 | Nivel de la **unidad médica de vigilancia epidemiológica** que atendió al paciente (primer, segundo o tercer nivel asistencial). |
| `medical_unit` | Nominal | 1–13 | **Tipo de institución** del Sistema Nacional de Salud que prestó la atención (IMSS, ISSSTE, Secretaría de Salud, etc.). |
| `intubed` | Binaria | 1 = sí, 2 = no | Si el paciente fue **conectado a un respirador** (intubación). Solo tiene sentido en pacientes hospitalizados. |
| `icu` | Binaria | 1 = sí, 2 = no | Si el paciente fue ingresado en la **Unidad de Cuidados Intensivos**. Igualmente, solo aplica a hospitalizados. |
| `date_died` | Fecha / centinela | fecha o `9999-99-99` | **Fecha de fallecimiento**. El código `9999-99-99` significa que el paciente **no falleció**. De aquí se deriva habitualmente la variable objetivo binaria. |

**Bloque 3 — Condición al ingreso**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `pneumonia` | Binaria | 1 = sí, 2 = no | Presencia de **neumonía** (inflamación de los sacos alveolares) en el momento de la atención. Es uno de los predictores más potentes de gravedad. |
| `pregnancy` | Binaria | 1 = sí, 2 = no | Si la paciente está **embarazada**. Solo aplica a mujeres. |

**Bloque 4 — Comorbilidades previas** *(todas binarias: 1 = sí, 2 = no)*

| Variable | Descripción |
|---|---|
| `diabetes` | Diabetes diagnosticada. |
| `copd` | **EPOC** (enfermedad pulmonar obstructiva crónica). |
| `asthma` | Asma. |
| `inmsupr` | **Inmunosupresión** (por enfermedad o tratamiento). |
| `hypertension` | Hipertensión arterial. |
| `cardiovascular` | Enfermedad del corazón o de los vasos sanguíneos. |
| `renal_chronic` | Enfermedad renal crónica. |
| `other_disease` | Cualquier **otra comorbilidad** no recogida en las categorías anteriores. |
| `obesity` | Obesidad. |
| `tobacco` | Consumo de tabaco. |

> **Codificaciones que hay que tratar antes de modelar:**
> * **`97`, `98`, `99` = dato ausente** en las variables codificadas → convertir a `NaN`.
> * **`date_died = 9999-99-99` = el paciente no falleció** → conviene derivar de aquí la variable binaria `died` (0/1) y **descartar la fecha**.
> * Los binarios vienen como **1/2** (no 0/1) → recodificar a 0/1 antes de interpretar coeficientes u *odds ratios*.
> * `pregnancy` solo aplica a mujeres (muchos faltantes en hombres).
>
> **Matiz importante sobre los tres códigos.** En el catálogo oficial mexicano, esos valores **no significan lo mismo**: `97` suele codificar «**no aplica**», `98` «**se ignora**» y `99` «**no especificado**». La distinción es relevante porque `97` marca una ausencia **estructural** —no es que se desconozca el dato, es que la pregunta carece de sentido: un hombre no puede estar embarazado, un paciente ambulatorio no puede ser intubado—, mientras que `98`/`99` indican **desconocimiento**. Convertirlos todos a `NaN` es aceptable, pero conviene saber que se está mezclando información de dos naturalezas distintas.

### <font color="steelblue">Advertencias metodológicas</font>

1. **El tamaño es sospechosamente redondo.** El número de filas coincide con **2²⁰ = 1.048.576**, que es exactamente el **límite de filas de una hoja de Excel**. Es muy probable que el fichero sea una **versión truncada** del registro original, no la población completa. Conviene mencionarlo: la muestra puede no ser representativa del total de casos registrados en México.

2. **Fuga de información: el gran riesgo de este conjunto.** Si el objetivo es predecir la **muerte** del paciente, variables como `intubed`, `icu` y `patient_type` **no son predictores legítimos**: describen decisiones clínicas tomadas *durante* la evolución del paciente, es decir, son **consecuencias** del deterioro, no causas anteriores a él. Un modelo que las incluya alcanzará métricas excelentes y será inútil en la práctica, porque en el momento en que se querría predecir el riesgo esa información **aún no existe**. Definir con precisión **en qué instante** se hace la predicción, y qué se sabe en ese instante, es la decisión más importante del proyecto.

3. **Ausencias no aleatorias (MNAR).** Los faltantes de `intubed` e `icu` están casi perfectamente determinados por `patient_type`, y los de `pregnancy` por `sex`. No son ausencias al azar: **la propia ausencia es información**. Imputar la moda en esos casos es un error conceptual; suele ser preferible restringir el análisis a la subpoblación pertinente, o codificar la ausencia como una categoría propia.

4. **Filtrar por diagnóstico antes de modelar.** El conjunto mezcla positivos, negativos e inconcluyentes. Al restringirse a los casos con COVID-19 confirmado (`classification` igual a 1, 2 o 3), quedan unos 391.979 registros.  Modelar sin filtrar mezcla poblaciones distintas y hace ininterpretable el resultado.

5. **Desequilibrio extremo del desenlace.** Los fallecimientos son una fracción reducida del total. Como en el cardiotocograma, la **exactitud global es una métrica engañosa**: lo relevante es el **recall** de la clase minoritaria y, dado el contexto clínico, el coste asimétrico de un falso negativo.

6. **Un registro administrativo no es un ensayo clínico.** Las asociaciones observadas (por ejemplo, entre `tobacco` y mortalidad) están sujetas a **sesgos de selección** —solo aparecen quienes acudieron al sistema de salud— y a **confusión** por variables no medidas. El modelo puede predecir bien sin que ninguna de sus «importancias» admita una lectura causal.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Definid el objetivo y el conjunto de variables ANTES de modelar**, y justificadlo (ver Fase 1–2).
2. **Cuidado con la fuga de datos.** No uséis como predictoras variables que son **consecuencia** del desenlace (ver Fase 2).
3. **Partición primero**, estratificada por la clase; el *test* solo se toca al final.
4. **Nada de fugas en el preprocesado:** imputación, escalado y **remuestreo** se ajustan **solo con el *train***, dentro de un **`Pipeline`** y rehaciéndose en cada pliegue de la CV.
5. **El equilibrado solo en *train*** (material 11). El *test* conserva la proporción real.
6. **Métricas acordes al desequilibrio:** *recall*/sensibilidad de la clase de riesgo, **F1**, **PR-AUC**, exactitud balanceada; la *accuracy* sola no vale.
7. **Reproducibilidad y honestidad:** `random_state` fijado; reportad también lo que no funcionó.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
# !pip -q install kagglehub imbalanced-learn gradio optuna scikit-learn shap
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Paso 1: descargar el dataset
path = kagglehub.dataset_download("meirnizri/covid19-dataset")
print("Ruta al dataset:", path)

# Paso 2: listar los archivos disponibles
archivos = os.listdir(path)
print("Archivos disponibles:")
for f in archivos:
    print(f"  · {f}")

# Paso 3: Cargar el dataset
df = pd.read_csv(os.path.join(path, archivos[0]))
print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

# <font color="steelblue">Fase 1 — Comprensión, definición del objetivo y EDA</font>

Aquí está la **decisión más importante** del proyecto: **¿qué predecimos?** El dataset no trae una columna objetivo; debéis crearla.

**Tareas obligatorias**
1. **Elegid y construid la variable objetivo.** Opciones razonables (elegid una y justificadla):
   * **Mortalidad (recomendada):** binaria `fallecido` derivada de `date_died` (`9999-99-99` → 0; cualquier fecha → 1).
   * **Alto riesgo (compuesta):** fallecimiento **o** ingreso en UCI **o** intubación. *(Si elegís esta, `icu` e `intubed` pasan a formar parte de `y` y **no** pueden ser predictoras.)*
   * **Hospitalización:** `patient_type` (domiciliaria vs hospitalización).
2. **(Opcional) Filtrad a casos positivos** (`classification` 1–3) si vuestro objetivo se refiere a enfermos confirmados; justificadlo.
3. **EDA:** distribución de la clase (¿cómo de desequilibrada?), distribución de `age`, prevalencia de comorbilidades, y **tasa de la clase de riesgo** según edad, sexo, neumonía, diabetes, hipertensión, etc.
4. **Conclusión:** 3–4 hallazgos que orienten el modelado.

> **A responder:** ¿por qué la *accuracy* sería engañosa con vuestro objetivo? ¿Qué métricas usaréis?

# <font color="steelblue">Fase 2 — Limpieza, fuga de datos y preprocesado</font>

**2A. Recodificación y faltantes (obligatorio)**
1. Sustituid **97/98/99 por `NaN`** en las variables codificadas.
2. Recodificad los **binarios 1/2 → 1/0** (1=sí → 1; 2=no → 0).
3. Decidid e implementad una **estrategia para los faltantes** (imputación; cuidado con `pregnancy`, que no aplica a hombres — quizá una categoría propia "no aplica").

**2B. Selección de variables y FUGA DE DATOS (clave)**

La introducción dice que el modelo debe predecir el riesgo **en el momento de dar positivo o antes**. Por tanto, no podéis usar variables que solo se conocen **después** del desenlace o que **son** el desenlace:

* **Excluir siempre:** `date_died` (define el objetivo de mortalidad).
* **Razonad y decidid:** `icu`, `intubed` y `patient_type` son **consecuencias** de la gravedad (ocurren durante el ingreso). Usarlas para predecir mortalidad es, en la práctica, **fuga**: el modelo "haría trampa". Lo realista es **excluirlas** y predecir con lo disponible al diagnóstico (demografía, comorbilidades, `pneumonia`, `classification`, `usmr`, `medical_unit`).
* Si vuestro objetivo es la **hospitalización**, entonces `patient_type` es el objetivo (no predictora) y `icu`/`intubed` siguen siendo posteriores.

**2C. Partición y `Pipeline`**
4. Separad `X`/`y`, **partición estratificada** *train*/*test*.
5. Preprocesado por tipo con `ColumnTransformer` dentro de `Pipeline` (escalar `age` para logística/SVM/kNN; los árboles no lo necesitan).

> **A responder:** ¿qué variables habéis descartado por fuga y por qué? Mostrad cómo cambia el rendimiento si (por curiosidad) incluís `icu`/`intubed`: veréis métricas "demasiado buenas" — esa es la señal de la fuga.

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias**
1. Entrenad y comparad **≥5 familias** del curso: **Regresión logística**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. Comparación con **CV estratificada** sobre el *train* y una métrica robusta al desequilibrio (**`average_precision`**/PR-AUC, **`f1`** o **`balanced_accuracy`**).
3. **Tabla** con media ± desviación por modelo y comentario.

> **Cómputo:** son ~1 millón de filas. Usad una **submuestra estratificada** (p. ej. 50–100k) para explorar y reentrenad el ganador sobre todo el *train*. Documentadlo.

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

La clase de riesgo (fallecidos) es **minoritaria**. Medid el efecto de tratarla (material **11. Equilibrando las muestras**), sobre los 2–3 mejores modelos de la Fase 3:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'` (o `sample_weight`).
3. **Sobremuestreo:** **SMOTE** (o Borderline-SMOTE / ADASYN).
4. **Submuestreo / híbrido:** `RandomUnderSampler` / **NearMiss** / SMOTEENN.

Reportad **recall de la clase de riesgo**, **F1**, **PR-AUC** y exactitud balanceada, y discutid el compromiso sensibilidad–precisión (en salud, perder un caso grave suele ser más costoso que una falsa alarma).

> **Sin fugas:** el remuestreo va **dentro** de un `Pipeline` de *imbalanced-learn* (solo el *train* de cada pliegue). **Nunca** el *test*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Tomad los **2–3 mejores** (modelo + equilibrado) y **optimizad sus hiperparámetros**.
2. `GridSearchCV`, `RandomizedSearchCV` **u Optuna**, con **CV estratificada** y la **misma métrica**.
3. El remuestreo/preprocesado va **dentro** del objeto de búsqueda (búsqueda sobre el `Pipeline`, prefijo `clf__`).
4. Reportad mejores hiperparámetros y la **mejora** frente a los valores por defecto.

> Con ~1M de filas, `RandomizedSearchCV`/Optuna sobre **submuestra** es lo eficiente; reentrenad el ganador al final.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores modelos optimizados con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`** (metamodelo sobre los base).
2. Comparad el conjunto frente al **mejor individual**: ¿mejora la métrica? ¿compensa el coste y la menor interpretabilidad?
3. **Combinad solo si aporta** mejora real (requisito: combinación *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación final, calibración e interpretación</font>

**Tareas obligatorias** (¡el *test* se usa una sola vez!)
1. Evaluad el **modelo final** en el *test*: **matriz de confusión**, `classification_report`, **recall de la clase de riesgo**, **F1**, **ROC-AUC** y **PR-AUC**.
2. **Umbral de decisión:** como el objetivo menciona la **probabilidad** de riesgo, no os quedéis en el 0.5 por defecto: elegid el umbral con la **curva precision-recall** según el coste clínico (priorizar **sensibilidad**). Mostrad cómo cambia la matriz de confusión.
3. **Calibración (recomendado):** comprobad si las probabilidades están bien calibradas (`CalibratedClassifierCV`, curva de fiabilidad). Importa si se reportan como "probabilidad de fallecer".
4. **Interpretabilidad:** `permutation_importance` y/o **SHAP**. ¿Pesan edad, neumonía, comorbilidades…? ¿Coincide con la literatura?
5. **Discusión crítica:** límites, sesgos (datos de 2020, posibles cambios poblacionales), qué mejoraríais.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** (preprocesado + modelo) con `joblib` y comprobad recarga + predicción sobre datos crudos.
2. **Función de predicción:** `predecir_riesgo(...)` que tome las variables del paciente (las **no-fuga**) y devuelva clase y **probabilidad** de riesgo.
3. **Interfaz interactiva:** app con **Gradio** (o `ipywidgets`) que recoja edad, sexo, comorbilidades y `pneumonia` y muestre el riesgo. En Colab, Gradio da un **enlace público**; incluidlo en la entrega.
4. (Opcional, nota extra) **Streamlit** o endpoint **FastAPI**.

> **Aviso clínico (obligatorio en la interfaz):** herramienta **educativa**, no es un diagnóstico ni sustituye el criterio médico. Datos de México, 2020.

# <font color="steelblue">Pistas y errores típicos</font>

* **El error estrella aquí: la fuga de datos.** Si tu modelo predice mortalidad casi perfecta, sospecha: probablemente has dejado `icu`, `intubed` o `patient_type` como predictoras. Repíte­lo sin ellas.
* **Recodifica antes de imputar.** Si no conviertes 97/98/99 en `NaN`, esos valores se tratan como números reales y contaminan todo.
* **`pregnancy`** no aplica a hombres: decide una política coherente (categoría "no aplica" o imputación informada).
* **No mires solo la *accuracy*.** Con ~7–10 % de fallecidos, predecir "sobrevive" siempre da *accuracy* alta e inútil.
* **Coste computacional:** ~1M de filas; submuestrea para explorar y reentrena el ganador.
* **Despliegue:** guarda el **Pipeline entero** y respeta el **orden/formato** de las columnas de `X`.

# <font color="steelblue">Referencias</font>

* Nizri, M. (2021). *COVID-19 Dataset*. Kaggle.
* Gobierno Federal de México — Dirección General de Epidemiología. *Casos COVID-19 en México* (datos.gob.mx).
* *Identification of high-risk COVID-19 patients using machine learning*. PMC, 2021.
* Rojas-García, M. et al. (2023). *Lethality risk markers by sex and age-group for COVID-19 in Mexico (XGBoost)*. BMC Infect. Dis.
* Cuadernos del curso: *Equilibrando las muestras*, *Boosting*, *Random Forest*, *Regresión logística binaria*.